In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
import pandas as pd
from src.paths import TRASNFORMED_DATA_DIR

df = pd.read_parquet(TRASNFORMED_DATA_DIR / 'tabular_data.parquet')
df

,rides_previous_672_hours,rides_previous_671_hours,rides_previous_670_hours,rides_previous_669_hours,rides_previous_668_hours,rides_previous_667_hours,rides_previous_666_hours,rides_previous_665_hours,rides_previous_664_hours,rides_previous_663_hours,...,rides_previous_7_hours,rides_previous_6_hours,rides_previous_5_hours,rides_previous_4_hours,rides_previous_3_hours,rides_previous_2_hours,rides_previous_1_hours,pickup_hour,pickup_location_id,target_rides_next_hour
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025-01-29,1,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,3.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-01-30,1,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,2025-01-31,1,0.0
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2025-02-01,1,0.0
4,0.0,1.0,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2025-02-02,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88289,0.0,3.0,2.0,4.0,3.0,1.0,2.0,1.0,3.0,3.0,...,4.0,2.0,1.0,1.0,2.0,1.0,2.0,2025-12-27,265,4.0
88290,2.0,5.0,2.0,1.0,6.0,0.0,3.0,0.0,3.0,2.0,...,4.0,8.0,0.0,3.0,3.0,1.0,0.0,2025-12-28,265,1.0
88291,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,8.0,0.0,...,8.0,7.0,1.0,6.0,2.0,7.0,2.0,2025-12-29,265,2.0
88292,1.0,1.0,0.0,1.0,2.0,0.0,1.0,3.0,0.0,0.0,...,3.0,3.0,3.0,4.0,0.0,6.0,1.0,2025-12-30,265,4.0


In [3]:
from datetime import datetime
from src.data_split import train_test_split

X_train, y_train, X_test, y_test = train_test_split(
    df=df,
    cutoff_data=datetime(2025,6,1,0,0,0),
    target_colum_name='target_rides_next_hour'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')


X_train.shape=(32226, 674)
y_train.shape=(32226,)
X_test.shape=(56068, 674)
y_test.shape=(56068,)


In [5]:
import numpy as np
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
import optuna

from src.model import get_pipeline

def objective(trial: optuna.trial.Trial) -> float:
    """
    Given a set of hyper-parameters, it trains a model and compute an average
    validation error based on a 'TimeSeriesSplit
    """
    # pick hyper-parameters
    hyperparams = {
        "metric": "mae",
        "verbose": -1,
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.2, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.2, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 3, 100),
    }

    tss = TimeSeriesSplit(n_splits=4)
    scores = []

    for train_index, val_index  in tss.split(X_train):

        # Split the data for training and validation
        X_train_ , X_val_ = X_train.iloc[train_index,:] , X_train.iloc[val_index ,:]
        y_train_, y_val_ = y_train.iloc[train_index] , y_train.iloc[val_index ]

        # train the model 
        pipeline = get_pipeline(**hyperparams)
        pipeline.fit(X_train_, y_train_)

        # validate the model
        y_pred= pipeline.predict(X_val_)
        mae = mean_absolute_error(y_val_, y_pred)

        scores.append(mae)

    return np.array(scores).mean()

In [6]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)

[I 2026-06-22 14:25:41,522] A new study created in memory with name: no-name-4fe9d00d-f0c0-4124-9a13-b1d661968cf3
[I 2026-06-22 14:26:26,953] Trial 0 finished with value: 5.497060340068324 and parameters: {'num_leaves': 239, 'feature_fraction': 0.8466745423591049, 'bagging_fraction': 0.6249589772683717, 'min_child_samples': 33}. Best is trial 0 with value: 5.497060340068324.
[I 2026-06-22 14:26:42,023] Trial 1 finished with value: 6.643329566188022 and parameters: {'num_leaves': 72, 'feature_fraction': 0.8000275535619454, 'bagging_fraction': 0.27071788687264803, 'min_child_samples': 82}. Best is trial 0 with value: 5.497060340068324.
[I 2026-06-22 14:26:50,895] Trial 2 finished with value: 6.5598292159771106 and parameters: {'num_leaves': 56, 'feature_fraction': 0.46309306792658056, 'bagging_fraction': 0.2788283393812286, 'min_child_samples': 61}. Best is trial 0 with value: 5.497060340068324.
[I 2026-06-22 14:27:21,622] Trial 3 finished with value: 6.58233750786024 and parameters: {'n

In [7]:
best_params = study.best_trial.params
print(f'{best_params=}')

best_params={'num_leaves': 239, 'feature_fraction': 0.8466745423591049, 'bagging_fraction': 0.6249589772683717, 'min_child_samples': 33}


In [8]:
pipeline = get_pipeline(**best_params)
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('functiontransformer', ...), ('temporalfeaturesengineer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](674,)","['rides_previous_672_hours','rides_previous_671_hours', 'rides_previous_670_hours',...,'rides_previous_1_hours','pickup_hour', 'pickup_location_id']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,674
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ave...002483BCBF7E0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


In [9]:
prediction = pipeline.predict(X_test)
test_mae = mean_absolute_error(y_test, prediction)
print(f'{test_mae=:.4f}')

test_mae=3.9764


In [10]:
from src.plot import plot_one_example

plot_one_example(
    features=X_test,
    target=y_test,
    example_id=2979,
    predictions=pd.Series(prediction)
)

In [14]:
from src.plot import plot_one_example

plot_one_example(
    features=X_test,
    target=y_test,
    example_id=3468,
    predictions=pd.Series(prediction)
)